![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Lab 1 (Solution): Build a CNN for MNIST

Build and train a **Convolutional Neural Network** from scratch in PyTorch to classify handwritten digits from the MNIST dataset.

**Goal:** Achieve over 99% accuracy on MNIST using a CNN with batch normalization, dropout, data augmentation, and a learning rate scheduler.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

plt.rcParams['figure.dpi'] = 120

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---
# Part 1 -- Data Loading and Exploration (GIVEN)

MNIST is a dataset of 70,000 handwritten digit images (28x28 pixels, grayscale). It is built into `torchvision`, so no manual download is needed.

In [ ]:
# --- GIVEN: Load MNIST ---
simple_transform = transforms.ToTensor()

train_dataset_raw = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=simple_transform
)
test_dataset_raw = torchvision.datasets.MNIST(
    root='./data', train=False, download=True, transform=simple_transform
)

print(f'Training samples: {len(train_dataset_raw)}')
print(f'Test samples:     {len(test_dataset_raw)}')

sample_img, sample_label = train_dataset_raw[0]
print(f'\nImage tensor shape: {sample_img.shape}  (channels x height x width)')
print(f'Pixel value range:  [{sample_img.min():.2f}, {sample_img.max():.2f}]')
print(f'Label: {sample_label}')
print(f'\nNumber of classes: {len(train_dataset_raw.classes)}')
print(f'Class labels: {train_dataset_raw.classes}')

In [ ]:
# --- GIVEN: Visualize a grid of samples ---
fig, axes = plt.subplots(4, 8, figsize=(12, 6))

for i, ax in enumerate(axes.flat):
    img, label = train_dataset_raw[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f'{label}', fontsize=10)
    ax.axis('off')

fig.suptitle('MNIST Training Samples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Part 2 -- Data Augmentation

We will apply random transformations to the training images so the model sees slightly different versions each epoch. This reduces overfitting.

## Task 1: Define Augmented Transforms

Fill in the `None` values below.

**Training transforms:** rotation, translation, convert to tensor, then normalize with MNIST stats.

**Test transforms:** just convert to tensor and normalize (no augmentation at test time).

In [ ]:
MNIST_MEAN = (0.1307,)
MNIST_STD  = (0.3081,)

train_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(MNIST_MEAN, MNIST_STD),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MNIST_MEAN, MNIST_STD),
])

In [ ]:
# --- GIVEN: Recreate datasets and DataLoaders with the new transforms ---
train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=train_transform
)
test_dataset = torchvision.datasets.MNIST(
    root='./data', train=False, download=True, transform=test_transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f'Train batches per epoch: {len(train_loader)}')
print(f'Test batches:            {len(test_loader)}')

---
# Part 3 -- Build the CNN

## Task 2: Define the Model

Fill in the `None` values to complete the CNN architecture.

**Conv block 1:** Input is 1-channel 28x28. After conv (same padding) + pool(2): output is 16 x 14 x 14.

**Conv block 2:** After conv (same padding) + pool(2): output is 32 x 7 x 7.

**Classifier:** Flatten, then fully connected layers with dropout.

In [ ]:
class MnistCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Conv block 1
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Conv block 2
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

## Task 3: Instantiate and Inspect

Create the model, move it to `device`, print it, count the trainable parameters, and verify the output shape with one batch.

In [ ]:
model = MnistCNN().to(device)
print(model)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTrainable parameters: {total_params:,}')

# Verify forward pass shape
sample_batch, _ = next(iter(train_loader))
sample_batch = sample_batch.to(device)
dummy_out = model(sample_batch)
print(f'Forward pass output shape: {dummy_out.shape}')  # should be [64, 10]

---
# Part 4 -- Training

## Task 4: Loss, Optimizer, and Scheduler

Fill in the `None` values:
- **Loss:** the standard loss for multi-class classification
- **Optimizer:** AdamW with learning rate and weight decay
- **Scheduler:** Cosine annealing over the full training run

In [ ]:
NUM_EPOCHS = 10

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=0.01,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
)

## Task 5: Training Loop

Fill in the `None` values inside the training loop. The structure is given.

Each epoch: train on all batches, then evaluate accuracy on the test set.

In [ ]:
train_losses = []
val_accuracies = []

for epoch in range(NUM_EPOCHS):
    # --- Training ---
    model.train()
    running_loss = 0.0
    num_batches = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        num_batches += 1

    avg_loss = running_loss / num_batches
    train_losses.append(avg_loss)

    # --- Validation ---
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100.0 * correct / total
    val_accuracies.append(val_acc)

    scheduler.step()

    print(f'Epoch [{epoch+1:2d}/{NUM_EPOCHS}]  '
          f'Loss: {avg_loss:.4f}  '
          f'Val Acc: {val_acc:.2f}%  '
          f'LR: {scheduler.get_last_lr()[0]:.6f}')

## Task 6: Plot Training Curves

Create a figure with two subplots side by side:
- Left: Training loss per epoch
- Right: Validation accuracy per epoch

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

epochs_range = range(1, NUM_EPOCHS + 1)

ax1.plot(epochs_range, train_losses, 'b-o', markersize=4)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Loss')
ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, val_accuracies, 'g-o', markersize=4)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Validation Accuracy')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
# Part 5 -- Evaluation

## Task 7: Test Set Accuracy

Evaluate the final model on the test set. Report overall accuracy as a percentage.

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_acc = 100.0 * correct / total
print(f'Final Test Accuracy: {test_acc:.2f}%')

## Task 8: Confusion Matrix

Collect all predictions and true labels from the test set. Use `sklearn.metrics.confusion_matrix` and `ConfusionMatrixDisplay` to show which digits are most commonly confused.

In [ ]:
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(range(10)))
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Task 9: Misclassified Examples

Find and display 10 misclassified images. Show each image with its true label and the model's predicted label.

In [ ]:
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

misclassified_idx = np.where(all_preds != all_labels)[0]
print(f'Misclassified: {len(misclassified_idx)} / {len(all_labels)}')

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    idx = misclassified_idx[i]
    img, true_label = test_dataset_raw[idx]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f'True: {all_labels[idx]}  Pred: {all_preds[idx]}',
                 fontsize=9, color='red')
    ax.axis('off')

fig.suptitle('Misclassified Examples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Conclusion

You built a CNN from scratch and trained it to classify MNIST digits with high accuracy. Key techniques include data normalization, cross-entropy loss, dropout, weight decay, batch normalization, a cosine learning rate schedule, convolutional layers, max pooling, data augmentation, and confusion matrix evaluation.

You now have the complete toolkit to tackle image classification problems end-to-end.